In [3]:
# --- Single channel NR algorithm ---
###############################################
# DATE    :  Nov 9th, 2021
# AUTHOR  :  Yi-Cheng Shane Hsu
###############################################
# DESCRIPTION :
#  - Frame Based Procedure
#  = Chunk Based Procedure
#  - Wiener Filter (WF)
#  - Minumum Mean Square Error (MMSE)
###############################################
!pip install opencv-python
!pip install import-ipynb
import rclpy
from rclpy.node import Node
from std_msgs.msg import Float32
import os
import cv2
import math
import wave
import scipy.special as sc
import pyaudio
import argparse
import numpy as np
import import_ipynb
import soundfile as sf
import IPython.display as ipd
import matplotlib.pyplot as plt

from numpy.linalg import pinv
from scipy.io import wavfile
from Array_Geometry_and_Time_Delay import GCC_PHAT_Dic

In [4]:
""" ID sounddevice """
p           = pyaudio.PyAudio()
info        = p.get_host_api_info_by_index(0)
numdevices  = info.get('deviceCount')

for i in range(0, numdevices):
    if (p.get_device_info_by_host_api_device_index(0, i).get('maxInputChannels')) > 0:
        print("Input Device id ", i, " - ", p.get_device_info_by_host_api_device_index(0, i).get('name'))

Input Device id  0  -  Microsoft 音效對應表 - Input
Input Device id  1  -  ReSpeaker 4 Mic Array (UAC1.0) 
Input Device id  2  -  麥克風排列 (Realtek(R) Audio)
Input Device id  3  -  麥克風 (Steam Streaming Microphone
Input Device id  4  -  麥克風 (NVIDIA Broadcast)


In [3]:
################ Parameters ###################
RESPEAKER_RATE        = 16000
RESPEAKER_CHANNELS    = 6
RESPEAKER_WIDTH       = 2
RESPEAKER_INDEX       = 2  # Input Device
SELECT_CHANNEL        = 1
CHANNEL_NUM           = 4
CHUNK_SIZE            = 512
MAXINT                = 2**15
SR                    = 16000
CHUNK_SIZE            = 512 # simulate the buffer of respeaker
NWIN                  = 512
HOP_SIZE              = 256
NFFT                  = 512
WIN                   = np.hamming(NWIN)
Freqs                 = np.arange(0, (NFFT/2)*SR+1, SR)

In [4]:
############### Audio Algorithm################
def SCNR(Mag_input, FrameCount, args, N, LambdaD, Gamma, G, NoiseCounter, mode=0):
    if FrameCount < args.avg_frame:
        SpeechFlag    = 0
        NoiseCounter  = 100
        N             = N + Mag_input
        LambdaD       = LambdaD + Mag_input ** 2
        return Mag_input, N, LambdaD, Gamma, G, NoiseCounter
        
    elif FrameCount == args.avg_frame:
        SpeechFlag    = 0
        NoiseCounter  = 100
        
        N             = N / args.avg_frame
        LamdaD        = LambdaD / args.avg_frame
        G             = np.ones(N.shape) * 0.001
        Gamma         = G
        return Mag_input, N, LambdaD, Gamma, G, NoiseCounter
        
    else:
        SpeechFlag, NoiseCounter = VAD(Mag_input, N, NoiseCounter, args.noisemargin)
        
    if (FrameCount > args.avg_frame) & (SpeechFlag == 0):
        N             = (args.noise_length * N + Mag_input) / (args.noise_length + 1)
        LambdaD       = (args.noise_length * LambdaD + (Mag_input ** 2)) / (1 + args.noise_length)
        
    gammaNew      = (Mag_input ** 2) / LambdaD
    ksi           = args.alpha * (G ** 2) * Gamma + (1 - args.alpha) * np.maximum(gammaNew - 1, 0)
    
    Gamma         = gammaNew
    nu            = Gamma * ksi / (1 + ksi)

    if mode == 0: # WF algo
        G           = (ksi/(1 + ksi))
        
    elif mode == 1: # log MMSE algo
        nu          = Gamma * ksi / (1 + ksi)
        G           = (ksi/(1 + ksi)) * np.exp(0.5 * sc.expn(1, nu))

    elif mode == 2: # MAP
        G           = (ksi + np.sqrt(ksi ** 2 + (1 + ksi) * (ksi / Gamma))) / (2*(1 + ksi))
        
    # --- Avoid nan and inf ---
    Indx         = np.isnan(G) | np.isinf(G)
    G[Indx]      = ksi[Indx] / (1 + ksi[Indx])
    Est_Mag      = G * Mag_input
    
    return Est_Mag, N, LambdaD, Gamma, G, NoiseCounter
    
def VAD(signal, noise, NoiseCounter, NoiseMargin, Hangover=6):
    SpectralDist = 20 * (np.log10(signal) - np.log10(noise))
    SpectralDist[SpectralDist < 0] = 0

    Dist = np.mean(SpectralDist)
    if (Dist < NoiseMargin):
        NoiseFlag = 1
        NoiseCounter = NoiseCounter + 1
    else:
        NoiseFlag = 0
        NoiseCounter = 0

    if (NoiseCounter > Hangover):
        SpeechFlag=0
    else:
        SpeechFlag=1

    return SpeechFlag, NoiseCounter

def RTF(NumOfMic, Freq, NumOfZone, TimeDelayDic):
    '''
    Calculate RTF in Freefield.
    Args:
        NumOfMic (int):         Number of Microphone
        Freq (list):            Frequency List
        NumOfZone (int):        Number Of Zone
        TimeDelayDic(np.array): shape (NumOfZone, NumOfMic - 1)
    Returns:
        RTF (np.array):         RTF / shape (NumOfMic, len(Freq), NumOfZone)
    '''
    
    d = np.ones((NumOfMic, len(Freq), NumOfZone),'complex')
    for j, ff in enumerate(Freq):
        for k in range(0, NumOfZone):
            d[1:, j, k] = np.exp(1j*2*math.pi*ff / 1*TimeDelayDic[k,:]).T
    return d

class Online_SRP_PHAT_v1:
    """
    Doing Localization by SRP-PHAT frame based. With VAD, calculate Rss through covariance substraction.
    Args:
        mode (str):             ex: n-Zone (n: int)
        MicPos (np.array):      Microphone Positon / shape (Dim=3, NumOfMic)
        fmin (int):             minimum frequency in bandlimited
        fmax (int):             maximum frequency in bandlimited
        fs (int):               sampling frequency
        NWIN (int):             number of sampling points in a window
        alpha_n (float):        recursive average parameter for noise covariance matrix
        alpha_x (float):        recursive average parameter for noisy covariance matrix
        r (float):              radius of UCA

    Returns:
        localization (float):   Azimuthal Angle (degree)
        
    """
    def __init__(self, mode, MicPos, fmin, fmax, fs, NWIN, alpha_n, alpha_x, r,):
        '''Setting'''
        self.NumOfMic        = MicPos.shape[1]
        self.MicPos          = MicPos
        self.alpha_n         = alpha_n
        self.alpha_x         = alpha_x
        self.mode            = mode
        
        '''Buffer'''
        self.Rnn             = []
        self.Rxx             = []
        
        '''FFT'''
        NFFT                 = int(2**math.ceil(math.log(NWIN, 2)))
        df                   = fs / NFFT
        Freq                 = np.arange(0, (NFFT/2 + 1)*df, df)
        self.Freq            = ((Freq[fmin <= Freq])[(Freq[fmin <= Freq]) <= fmax]).tolist()
        self.Freq_index      = (fmin <= Freq) * (Freq <= fmax)
        
        '''Mode'''
        if 'Zone' in mode:
            self.NumOfZone   = int(''.join(filter(str.isdigit, mode)))
        else:
            print('Cannot Find ' + mode)
            
        '''GCC-PHAT Dictionary'''
        if os.path.exists('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz') == 1:
            file = np.load('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz',  allow_pickle=True)
            TimeDelayDic = file['TimeDelayDic']
        else:
            TimeDelayDic = GCC_PHAT_Dic(self.NumOfZone, r, fs, NWIN, MicPos)
            np.savez('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz', TimeDelayDic = TimeDelayDic)
        
        '''RTF'''
        self.d               = RTF(self.NumOfMic, self.Freq, self.NumOfZone, TimeDelayDic)
        
    def step_frame(self, frame, VAD):
        """
        Online SRP-PHAT in framewise fashion.
        Args:
            frame:                  shape (NumOfMic, len(Freq))
            VAD:                    1 or 0
        Returns:
            localization (float):   Azimuthal Angle (degree)
        """
        frame_band_limited = frame[:, self.Freq_index]
        if VAD == 1:
            self._calculate_noisy_covariance_matrix(frame_band_limited)
            localization = self._SRP_PHAT(frame_band_limited)
            return localization
        else:
            self._calculate_noisy_covariance_matrix(frame_band_limited)
            self._calculate_noise_covariance_matrix(frame_band_limited)
            return -1 #None
    
    def _SRP_PHAT(self, frame):
        Rss = self.Rxx - self.Rnn
        R = Rss / abs(Rss)
        M = np.einsum('ijk,ilj,ljk->k', np.conj(self.d), R, self.d)
        return np.argmax(M)
    
    def _calculate_noise_covariance_matrix(self, frame):
        if len(self.Rnn) == 0:
            self.Rnn = np.einsum('ij,kj->ikj', frame, np.conj(frame))
        else:
            self.Rnn = self.alpha_n*self.Rnn + (1 - self.alpha_n)*np.einsum('ij,kj->ikj', frame, np.conj(frame))
    def _calculate_noisy_covariance_matrix(self, frame):
        Rxx_new = np.einsum('ij,kj->ikj', frame, np.conj(frame))
        if len(self.Rxx) == 0:
            self.Rxx = Rxx_new
        else:
            self.Rxx = self.alpha_x*self.Rxx + (1 - self.alpha_x)*Rxx_new
        
def UCA_Geometry(MicNum, r):
    theta               = 360 / MicNum
    MicPos_x            = np.cos(np.radians(np.arange(0, 360, theta)))
    MicPos_y            = np.sin(np.radians(np.arange(0, 360, theta)))
    MicPos_z            = np.zeros(MicNum)
    MicPos              = r * np.vstack((MicPos_x, MicPos_y, MicPos_z))
    return MicPos


        
class Online_Localization_TDOA:
    """
    Calculate TDOA first by GCC-PHAT. Then solve kappa in the equation R*kappa=toque(TDOA).
    Args:
        fs (int):               sampling frequency
        NWIN (int):             number of sampling points in a window
        MicPos (np.array):      Microphone Positon / shape (Dim=3, NumOfMic)
        r (float):              radius of UCA
        lambda_ (float):        recursive average parameter for cross-spectral density
    Returns:
        localization (float):   Azimuthal Angle (degree)
    """
    def __init__(self, fs, NWIN, MicPos, r, lambda_):
        '''Setting'''
        self.fs              = fs
        self.max_delay_pt    = int(round(2 * r / 343.0 * fs, 0))
        self.R               = (MicPos[0: 2, 1:] - MicPos[0: 2, 0: 1]).T  # shape (NumOfMic - 1, Dim=2)
        self.lambda_         = lambda_
        self.G               = []
        
        '''FFT'''
        self.NFFT            = int(2**math.ceil(math.log(NWIN, 2)))
        
    def GCC_PHAT(self, frame,):
        """
        Online GCC-PHAT in framewise fashion.
        Args:
            frame:                  shape (NumOfMic, len(Freq))
        Returns:
            Differ_time(np.array):  TDOA / shape (NumOfMic - 1)
        """
        '''Cross-Spectrum Density'''
        if len(self.G) == 0:
            self.G = np.einsum('j,ij->ij', frame[0, :], np.conj(frame[1:, :]))
        else:
            self.G = self.lambda_*self.G + (1 - self.lambda_)*np.einsum('j,ij->ij', frame[0, :], np.conj(frame[1:, :]))
        # G / shape (NumOfMic - 1, freq)
        
        '''Whitening'''
        G_whit = self.G / np.maximum(abs(self.G), self.NFFT*1e-3)
        
        '''IFFT'''
        g = np.array([np.fft.irfft(G_whit[M, :]) for M in range(G_whit.shape[0])])
        r = np.concatenate((g[:, self.NFFT//2:], g[:, : self.NFFT//2]), axis=1)
        
        '''Restriction'''
        r = r[:, self.NFFT//2 - self.max_delay_pt - 1 : self.NFFT//2 + self.max_delay_pt]
        
        '''Calculate TDOA'''
        INDEX = np.argmax(r, axis=1)
        Differ_point = INDEX - self.max_delay_pt
        Differ_time = Differ_point / self.fs
        Differ_dis = Differ_time * 343.0
        return Differ_time
    
    def _TDOA_localization_solve(self, TDOA):
        kappa = pinv(np.dot(self.R.T, self.R)).dot(self.R.T).dot(TDOA)
        
        azimuth = (math.degrees(np.arctan(kappa[1] / kappa[0])) + (kappa[0] < 0)*180 + 360) % 360
        elevation = math.degrees(np.arccos((kappa[0]**2 + kappa[1]**2)**(1/2)))
        return azimuth, elevation
        
    def step_frame(self, frame,):
        """
        Online localization in framewise fashion.
        Args:
            frame:                  shape (NumOfMic, len(Freq))
        Returns:
            localization (float):   Azimuthal Angle (degree)
        """
        TDOA = self.GCC_PHAT(frame)
        azimuth, elevation = self._TDOA_localization_solve(TDOA)
        return azimuth
    
class Online_SRP_PHAT_without_VAD:
    """
    Doing Localization by SRP-PHAT frame based. With VAD, calculate Rss through covariance substraction.
    Args:
        mode (str):             ex: n-Zone (n: int)
        MicPos (np.array):      Microphone Positon / shape (Dim=3, NumOfMic)
        fmin (int):             minimum frequency in bandlimited
        fmax (int):             maximum frequency in bandlimited
        fs (int):               sampling frequency
        NWIN (int):             number of sampling points in a window
        alpha_x (float):        recursive average parameter for noisy covariance matrix
        r (float):              radius of UCA

    Returns:
        localization (float):   Azimuthal Angle (degree)
        
    """
    def __init__(self, mode, MicPos, fmin, fmax, fs, NWIN, alpha_x, r,):
        '''Setting'''
        self.NumOfMic        = MicPos.shape[1]
        self.MicPos          = MicPos
        self.alpha_x         = alpha_x
        self.mode            = mode
        
        '''Buffer'''
        self.Rxx             = []
        
        '''FFT'''
        NFFT                 = int(2**math.ceil(math.log(NWIN, 2)))
        df                   = fs / NFFT
        Freq                 = np.arange(0, (NFFT/2 + 1)*df, df)
        self.Freq            = ((Freq[fmin <= Freq])[(Freq[fmin <= Freq]) <= fmax]).tolist()
        self.Freq_index      = (fmin <= Freq) * (Freq <= fmax)
        
        '''Mode'''
        if 'Zone' in mode:
            self.NumOfZone   = int(''.join(filter(str.isdigit, mode)))
        else:
            print('Cannot Find ' + mode)
            
        '''GCC-PHAT Dictionary'''
        if os.path.exists('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz') == 1:
            file = np.load('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz',  allow_pickle=True)
            TimeDelayDic = file['TimeDelayDic']
        else:
            TimeDelayDic = GCC_PHAT_Dic(self.NumOfZone, r, fs, NWIN, MicPos)
            np.savez('TimeDelayDic_'+str(r)+'_'+str(self.NumOfZone)+'_'+str(self.NumOfMic)+'.npz', TimeDelayDic = TimeDelayDic)
        
        '''RTF'''
        self.d               = RTF(self.NumOfMic, self.Freq, self.NumOfZone, TimeDelayDic)
        
    def step_frame(self, frame):
        """
        Online SRP-PHAT in framewise fashion.
        Args:
            frame:                  shape (NumOfMic, len(Freq))
        Returns:
            M_kappa (np.array):     shape (12)
        """
        frame_band_limited = frame[:, self.Freq_index]
        self._calculate_noisy_covariance_matrix(frame_band_limited)
        localization = self._SRP_PHAT(frame_band_limited)
        return localization
    
    def _SRP_PHAT(self, frame):
        R = self.Rxx / abs(self.Rxx)
        M = abs(np.einsum('ijk,ilj,ljk->k', np.conj(self.d), R, self.d))
        index = np.argmax(M)
        Angle = index * 360 / self.NumOfZone
        return Angle

    def _calculate_noisy_covariance_matrix(self, frame):
        Rxx_new = np.einsum('ij,kj->ikj', frame, np.conj(frame))
        if len(self.Rxx) == 0:
            self.Rxx = Rxx_new
        else:
            self.Rxx = self.alpha_x*self.Rxx + (1 - self.alpha_x)*Rxx_new

In [5]:
zone_num            = 12
mode2               = str(zone_num) + '-Zone'
r                   = 0.0325
fmin                = 0
fmax                = math.floor(343.0 / (2 * (2*r**2 - 2*(r**2)*np.cos(np.radians(360/CHANNEL_NUM)))**(1/2)))
# alpha_n             = 0.98
alpha_x             = 0.2
# covar_threshold     = 10000

In [6]:
""" Algorithm's parameters setting """
parser            = argparse.ArgumentParser()
parser.add_argument('--noise-length', type=int, dest='noise_length', default=4)
parser.add_argument('--alpha', type=float, dest='alpha', default=0.95)
parser.add_argument('--avg-frame', type=int, dest='avg_frame', default=6)
parser.add_argument('--noisemargin', type=int, dest='noisemargin', default=6)
args              = parser.parse_args(['--noise-length', '4', 
                                       '--alpha', '0.98', 
                                       '--avg-frame', '6', 
                                       '--noisemargin', '6'])

In [ ]:
Mag_buffer1     = np.zeros((HOP_SIZE,CHANNEL_NUM))
Mag_buffer2     = np.zeros((HOP_SIZE,CHANNEL_NUM))
Output_buffer   = np.zeros((CHUNK_SIZE,CHANNEL_NUM))
N               = np.zeros((int(NWIN/2+1)))
LambdaD         = np.zeros((int(NWIN/2+1)))
G               = np.zeros((int(NWIN/2+1)))
Gamma           = np.zeros((int(NWIN/2+1)))
NoiseCounter    = 0
Angle           = 0
mode            = 2
MicPos          = UCA_Geometry(CHANNEL_NUM, r)
lambda_         = 0.8
degree_         = 360 // zone_num
    
""" localization initialization """
# online_TDOA         = Online_Localization_TDOA(SR, NWIN, MicPos, r, lambda_)
online_SRP_PHAT     = Online_SRP_PHAT_without_VAD(mode2, MicPos, fmin, fmax, SR, NWIN, alpha_x, r,)

""" Streaming setting """
p               = pyaudio.PyAudio()
stream          = p.open(format=pyaudio.paInt16, rate=RESPEAKER_RATE, 
                         channels=RESPEAKER_CHANNELS, input=True, 
                         frames_per_buffer=CHUNK_SIZE, input_device_index=RESPEAKER_INDEX)

player          = p.open(format=pyaudio.paInt16, rate=RESPEAKER_RATE,
                         channels=1, output=True, frames_per_buffer=CHUNK_SIZE)

FrameBuffer     = np.zeros((CHUNK_SIZE*200, CHANNEL_NUM)) #Dump wav file
FrameBuffer2    = np.zeros((CHUNK_SIZE*200, CHANNEL_NUM)) #Dump wav file

print("Start recording...")
Count           = 0

try:
    
    # setting figure size
    newImageInfo = (500, 500, 3)
    img = np.zeros(newImageInfo, np.uint8)
    # Fill image with gray color(set each pixel to gray)
    img[:] = (128, 128, 128)

    cv2.circle(img, (250,250), 200, (255, 255, 255), -1)
    cv2.circle(img, (250,250), 200, (0, 0, 0), thickness = 5)
    center_x = 250
    center_y = 250
    radius = 200
    margin = 200
    pt1 = []
    # plot zone line
    for i in range(zone_num):
        x1 = center_x + (radius - margin) * math.cos((i + 0.5) * degree_ * np.pi / 180.0)
        y1 = center_y + (radius - margin) * math.sin((i + 0.5) * degree_ * np.pi / 180.0)
        pt1.append((int(x1), int(y1)))

        x2 = center_x + (radius - 5) * math.cos((i + 0.5) * degree_ * np.pi / 180.0)
        y2 = center_y + (radius - 5) * math.sin((i + 0.5) * degree_ * np.pi / 180.0)

        cv2.line(img, pt1[i], (int(x2), int(y2)), (100, 100, 100), thickness = 2)
        num = str(i + 1)
        font = cv2.FONT_HERSHEY_SIMPLEX
        text_x = center_x + (radius - 40) * math.cos(i * degree_ * np.pi / 180.0)
        text_y = center_y + (radius - 40) * math.sin(i * degree_ * np.pi / 180.0)
        if (i == 0):
            cv2.putText(img, str(1), (int(text_x) - 15, int(text_y) + 10), font, 1, (0, 0, 0), 2)# zone number
        else :
            num = zone_num + 1 - i
            if (num >= 10):
                cv2.putText(img, str(num), (int(text_x) - 20, int(text_y) + 10), font, 1, (0, 0, 0), 2)# zone number
            else:
                cv2.putText(img, str(num), (int(text_x) - 15, int(text_y) + 10), font, 1, (0, 0, 0), 2)# zone number

    # algo.


    rclpy.init(args=None)
    ros_node = rclpy.create_node('srp_phat_left_node')
    angle_pub = ros_node.create_publisher(Float32, '/sound_angle_left', 10)
    while True:
        # --- get audio chunk ---
        stringAudioData     = stream.read(CHUNK_SIZE, exception_on_overflow=False)
        
        audioData           = []
        for i in range(CHANNEL_NUM):
            audioData.append(np.frombuffer(stringAudioData, dtype=np.int16)[SELECT_CHANNEL+i::RESPEAKER_CHANNELS])
        
        audioData           = np.array(audioData)
        
        normalizedData      = audioData / MAXINT # Normalize
        
        data                = np.concatenate((Mag_buffer1, normalizedData.T))
        Mag_buffer1         = data[HOP_SIZE*2:]
        
        for i in range(2):
            start             = i * HOP_SIZE
            signal            = data[start:start + NWIN, :]
            signal            = signal * WIN[:,None]

            # --- FFT ---
            Y          = np.fft.fft(signal, axis=0)
            Y_local    = Y[0:int(np.fix(len(Y)/2))+1].copy()
            YPhase     = np.angle(Y[0:int(np.fix(len(Y)/2))+1])
            Y          = np.abs(Y[0:int(np.fix(len(Y)/2))+1])
            
            """ Main Algorithm """
            Y_est, N, LambdaD, Gamma, G, NoiseCounter= SCNR(Y[:,0].T, FrameCount=Count, args=args, N=N, LambdaD=LambdaD, Gamma=Gamma, G=G, NoiseCounter=NoiseCounter, mode=mode)
            
            Angle      = online_SRP_PHAT.step_frame(frame=Y_local.T)
#             Angle_old   = Angle
            
#             if NoiseCounter == 0:
#                 Angle            = online_TDOA.step_frame(frame=Y_local.T)
            
            temp = np.copy(img)
    
            if (NoiseCounter == 0):
                cv2.ellipse(temp,(250,250),(195, 195), (360 - (degree_ / 2) - Angle), 0, degree_, (0,255,255), -1)
            
            
                angle_rad = float(Angle * math.pi / 180.0)
                msg = Float32()
                msg.data = angle_rad
                angle_pub.publish(msg)
                # print(f"Published Left Angle: {angle_rad:.3f} rad")
                

                rclpy.spin_once(ros_node, timeout_sec=0)
                
            cv2.imshow('zone', temp)
            cv2.waitKey(2)
            
            # --- IFFT ---
#             Y_est      = Y_est * np.exp(1j * YPhase[:,0])
#             Y_est      = np.concatenate((Y_est, np.flipud(np.conj(Y_est[1:-1]))))
            
            Y          = Y * np.exp(1j * YPhase)
            Y          = np.concatenate((Y, np.flipud(np.conj(Y[1:-1]))))
            y          = np.real(np.fft.ifft(Y, axis=0))
            
            # --- Overlap and Add ---
            Output_buffer[HOP_SIZE*i:HOP_SIZE*(i+1)] = Mag_buffer2 + y[0:HOP_SIZE]
            Mag_buffer2     = y[HOP_SIZE:]
            
        
        outputData          = Output_buffer
        up                  = normalizedData
        
        audioData           = np.array(np.round_(outputData[:,3]*MAXINT), dtype=np.int16)
        stringAudioData     = audioData.tobytes()
        player.write(stringAudioData, CHUNK_SIZE)
        
        """ Dump wav file """
        if  Count < 200:
            FrameBuffer[CHUNK_SIZE*Count:CHUNK_SIZE*(Count+1),:]  = outputData
            FrameBuffer2[CHUNK_SIZE*Count:CHUNK_SIZE*(Count+1),:] = up.T
        
        """ Save wav file """
        if Count == 200 - 1:
            sf.write('denoise.wav', FrameBuffer, RESPEAKER_RATE) 
            sf.write('original.wav', FrameBuffer2, RESPEAKER_RATE)
            print('Recorded')
        
        
        Count              += 1
    
finally:
    stream.stop_stream()
    stream.close()
    p.terminate()
    ros_node.destroy_node()
    rclpy.shutdown()


Start recording...
Recorded


OSError: Stream not open